# Data Cleaning & Transformation

This notebook transforms the raw Bangalore Rapido Ride dataset into a clean, structured,
and analysis-ready format.

Key steps:
- Handling missing values using business logic
- Converting date and time columns
- Removing duplicates
- Standardizing categorical values
- Feature engineering

Special attention is given to cancelled rides, where missing values represent valid business scenarios.

## Loading Raw Dataset

The raw dataset is loaded from the extraction phase.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/rides_data.csv")
df.head()

,services,date,time,ride_status,source,destination,duration,ride_id,distance,ride_charge,misc_charge,total_fare,payment_method
0,cab economy,2024-07-15,08:30:40.542646,completed,Balagere Harbor,Harohalli Nagar,39,RD3161218751875354,27.21,764.83,31.51,796.34,Amazon Pay
1,auto,2024-07-05,23:36:51.542646,completed,Basavanagudi 3rd Block,Bikasipura 1st Stage,89,RD8171514284594096,34.03,314.83,49.52,364.35,Paytm
2,auto,2024-07-23,11:05:37.542646,cancelled,Babusapalya Cove,Kothaguda Terrace,25,RD9376481122237926,20.24,NaN,NaN,NaN,NaN
3,cab economy,2024-06-24,08:45:10.542646,completed,Mahadevapura Mews,Kanakapura Arc,89,RD3676889143182765,31.17,484.73,15.84,500.57,QR scan
4,cab economy,2024-07-15,00:26:44.542646,completed,Ganganagar Cove,Basaveshwaranagar Colony,95,RD6639410275948084,27.21,663.50,14.13,677.63,Amazon Pay


## Data Inspection

We examine the dataset to identify:
- Missing values
- Incorrect data types
- Data inconsistencies

This step defines the cleaning strategy.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   services        50000 non-null  object 
 1   date            50000 non-null  object 
 2   time            50000 non-null  object 
 3   ride_status     50000 non-null  object 
 4   source          50000 non-null  object 
 5   destination     50000 non-null  object 
 6   duration        50000 non-null  int64  
 7   ride_id         50000 non-null  object 
 8   distance        50000 non-null  float64
 9   ride_charge     44964 non-null  float64
 10  misc_charge     44964 non-null  float64
 11  total_fare      44964 non-null  float64
 12  payment_method  44964 non-null  object 
dtypes: float64(4), int64(1), object(8)
memory usage: 5.0+ MB


In [ ]:
df.isnull().sum()

,0
services,0
date,0
time,0
ride_status,0
source,0
destination,0
duration,0
ride_id,0
distance,0
ride_charge,5036


## Handling Missing Values

Missing values are observed in:
- ride_charge
- misc_charge
- total_fare
- payment_method

These missing values correspond to cancelled rides.

Handling strategy:
- Cancelled rides → fare values set to 0, payment set to "Not Applicable"
- Completed rides → total_fare recalculated as ride_charge + misc_charge

This ensures logical consistency and avoids data loss.

In [ ]:
cancelled_mask = df['ride_status'] == 'cancelled'
# Set values for cancelled rides
df.loc[cancelled_mask, ['ride_charge', 'misc_charge', 'total_fare']] = 0
df.loc[cancelled_mask, 'payment_method'] = "Not Applicable"

In [ ]:
# Recalculate total fare for completed rides
completed_mask = df['ride_status'] == 'completed'
df.loc[completed_mask, 'total_fare'] = (
    df.loc[completed_mask, 'ride_charge'] +
    df.loc[completed_mask, 'misc_charge']
)
df['revenue'] = df['total_fare']

## Date Conversion

The date column is converted into datetime format
to enable time-based analysis.

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
# Remove invalid dates
df = df.dropna(subset=['date'])

## Time Column Conversion

The time column is converted into datetime format to enable time-based analysis.

Instead of converting to a plain time object, the datetime format is retained
to allow extraction of features such as hour and minute.

This improves analytical flexibility and supports efficient grouping operations.

In [ ]:
df['time'] = pd.to_datetime(df['time'], errors='coerce')

/tmp/ipykernel_8475/146901932.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time'] = pd.to_datetime(df['time'], errors='coerce')


## Removing Duplicates

Duplicate rows are removed to ensure data accuracy.

In [ ]:
df = df.drop_duplicates(subset=['ride_id'], keep='first')

## Data Standardization

Categorical values are standardized to ensure consistency.

In [ ]:
# Standardize all categorical and location strings
cols_to_fix = ['services', 'ride_status', 'payment_method', 'source', 'destination']

for col in cols_to_fix:
    df[col] = df[col].astype(str).str.lower().str.strip()

# Map the service names for consistency
df['services'] = df['services'].replace({
    'bike lite': 'bike_lite',
    'cab economy': 'cab_economy'
})

## Feature Engineering

New features are created to enhance analysis:

- revenue → total fare per ride
- year, month, day → time-based analysis
- hour → peak demand identification
- day_of_week → weekday vs weekend analysis
- peak_hour → high demand indicator
- completed → ride success indicator

In [ ]:
# Date features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.day_name()

# Time features (FIXED)
df['hour'] = df['time'].dt.hour
df['minute'] = df['time'].dt.minute


# Peak hour
df['peak_hour'] = df['hour'].apply(
    lambda x: 1 if x in [8,9,10,17,18,19] else 0
)

# Completed flag
df['completed'] = df['ride_status'].apply(
    lambda x: 1 if x == 'completed' else 0
)

# Weekend flag
df['is_weekend'] = df['day_of_week'].apply(
    lambda x: 1 if x in ['Saturday', 'Sunday'] else 0
)

## Anomaly Detection: Speed & Distance
We filter out 'completed' rides that are physically impossible, such as those with
implausible speeds (e.g., > 80 km/h in Bangalore traffic).

In [ ]:
# Calculate speed in km/h (distance in km / duration in hours)
df['avg_speed_kmh'] = df['distance'] / (df['duration'] / 60)
# Filter out impossible scenarios
# 1. Completed rides with 0 distance
# 2. Rides with speed > 120 km/h (unlikely for city bike/auto)
# 3. Rides with 0 duration
invalid_rides = df[
    (df['ride_status'] == 'completed') &
    ((df['avg_speed_kmh'] > 120) | (df['distance'] <= 0) | (df['duration'] <= 0))
]

df = df.drop(invalid_rides.index)
print(f"Dropped {len(invalid_rides)} anomalous records.")

Dropped 2003 anomalous records.


## Advanced Feature Engineering

Additional features are created to improve analytical depth:
- fare_per_km
- ride_efficiency (distance/duration)

In [ ]:
# Fare per km
df['fare_per_km'] = df['revenue'] / df['distance']
df['ride_efficiency'] = df['distance'] / df['duration']

## Post-Cleaning Validation

After handling missing values, we verify that:

- No critical null values remain
- All numerical fields are complete
- Dataset is consistent for analysis

This ensures reliability of downstream analysis and dashboard metrics.

In [ ]:
df.isnull().sum()

,0
services,0
date,0
time,0
ride_status,0
source,0
destination,0
duration,0
ride_id,0
distance,0
ride_charge,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47997 entries, 0 to 49999
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   services         47997 non-null  object        
 1   date             47997 non-null  datetime64[ns]
 2   time             47997 non-null  datetime64[ns]
 3   ride_status      47997 non-null  object        
 4   source           47997 non-null  object        
 5   destination      47997 non-null  object        
 6   duration         47997 non-null  int64         
 7   ride_id          47997 non-null  object        
 8   distance         47997 non-null  float64       
 9   ride_charge      47997 non-null  float64       
 10  misc_charge      47997 non-null  float64       
 11  total_fare       47997 non-null  float64       
 12  payment_method   47997 non-null  object        
 13  revenue          47997 non-null  float64       
 14  year             47997 non-null  int32     

In [ ]:
df.to_csv("../data/processed/cleaned_rides.csv", index=False)

## Data Quality Summary

- 10% missing values due to cancelled rides (handled using business logic)
- Fare inconsistencies corrected using recalculation
- Invalid and unrealistic records removed
- Categorical data standardized
- New features engineered for analysis

Final dataset is clean, consistent, and analysis-ready.